In [2]:
import pandas as pd
import numpy as np

In [ ]:
# starting_elo_df = pd.read_csv("starting_elo.csv")

In [ ]:
# df = pd.read_csv("cleaned_data.csv", index_col=0)

In [ ]:
# elo_container = starting_elo_df.iloc[0].to_dict()

In [ ]:
# elo_container

In [7]:
df["gameDateTimeEst"] = pd.to_datetime(df["gameDateTimeEst"])

In [8]:
def season_column(df):
    df["season"] = np.where(df["gameDateTimeEst"].dt.month >= 10, df["gameDateTimeEst"].dt.year + 1, df["gameDateTimeEst"].dt.year)
    return df

In [ ]:
# df = season_column(df)
# df

,gameId,gameDateTimeEst,teamName,teamId,home,win,points,opponentScore,possessions,eFG,TO%,OREB%,FTR,off_rating,def_rating,net_rating,season
0,28600001,1986-10-31 20:00:00,Thunder,1610612760,0,1.0,127,110,102.2976,0.622093,0.224834,0.448276,0.279070,124.147585,107.529404,16.618181,1987
1,28600001,1986-10-31 20:00:00,Trail Blazers,1610612757,1,0.0,110,127,100.1088,0.506494,0.199783,0.272727,0.480519,109.880450,126.861974,-16.981524,1987
2,28600002,1986-10-31 20:00:00,Pacers,1610612754,0,0.0,104,108,94.4640,0.469880,0.148205,0.318182,0.421687,110.094851,114.329268,-4.234417,1987
3,28600002,1986-10-31 20:00:00,76ers,1610612755,1,1.0,108,104,100.0320,0.523810,0.249920,0.461538,0.357143,107.965451,103.966731,3.998720,1987
4,28600003,1986-10-31 20:00:00,Wizards,1610612764,0,0.0,102,120,98.7264,0.435294,0.222838,0.400000,0.423529,103.315830,121.548036,-18.232205,1987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102671,42500301,2026-05-19 20:00:00,Knicks,1610612752,1,1.0,115,104,105.6768,0.534091,0.160868,0.250000,0.363636,108.822372,98.413275,10.409096,2026
102672,42500312,2026-05-20 20:30:00,Spurs,1610612759,0,0.0,113,122,92.6208,0.583333,0.226731,0.400000,0.202381,122.002833,131.719873,-9.717040,2026
102673,42500312,2026-05-20 20:30:00,Thunder,1610612760,1,1.0,122,113,92.6976,0.547872,0.097090,0.369565,0.255319,131.610743,121.901754,9.708989,2026
102674,42500302,2026-05-21 20:00:00,Cavaliers,1610612739,0,0.0,93,109,85.5168,0.443750,0.093549,0.270833,0.400000,108.750561,127.460335,-18.709774,2026


In [10]:
def calculate_new_elo(old_elo, m, s, e, k=20):
    new_elo = old_elo + k * m * (s - e)

    return new_elo

In [11]:
def margin_of_victory(mov, elo_diff):
    m = ((mov + 3)**.8) / (7.5 + .006 * (elo_diff))
    
    return m

In [12]:
def off_season_regression(elo_container):
    for (team_name, current_elo) in elo_container.items():
        regressed_elo = (current_elo * .75) + (1505 * 0.25)

        elo_container[team_name] = regressed_elo
    return elo_container


In [13]:
def elo_calculation(df, elo_container):
    df = df.sort_values(by=["gameDateTimeEst", "gameId", "home"])
    pre_game_elos = []

    current_season = df["season"].iloc[0]
    for game_id, team_df in df.groupby("gameId", sort=False):

        game_season = team_df["season"].iloc[0]
        if current_season != game_season:
            off_season_regression(elo_container)
            current_season = game_season


        if len(team_df) != 2:
            continue


        home_row = team_df[team_df["home"] == 1].iloc[0]
        away_row = team_df[team_df["home"] == 0].iloc[0]

        home_name = home_row["teamName"]
        away_name = away_row["teamName"]

        home_pre = elo_container.get(home_name, 1500)
        away_pre = elo_container.get(away_name, 1500)

        pre_game_elos.extend([away_pre, home_pre])

        d = (home_pre + 100) - away_pre
        expected_home = 1 / (1 + 10 ** (-d / 400))
        expected_away = 1 - expected_home

        if home_row["win"] == 1:
            mov = home_row["points"] - home_row["opponentScore"]
            elo_diff = home_pre - away_pre
            home_s, away_s = 1, 0
        else:
            mov = away_row["points"] - away_row["opponentScore"]
            elo_diff = away_pre - home_pre
            home_s, away_s = 0, 1


        m = margin_of_victory(mov, elo_diff)

        home_post = calculate_new_elo(home_pre, m, home_s, expected_home)
        away_post = calculate_new_elo(away_pre, m, away_s, expected_away)


        elo_container[home_name] = round(home_post, 2)
        elo_container[away_name] = round(away_post, 2)

    df["pre_game_elo"] = pre_game_elos
    return df



In [ ]:
# df = elo_calculation(df, elo_container)

In [15]:
df.head()

,gameId,gameDateTimeEst,teamName,teamId,home,win,points,opponentScore,possessions,eFG,TO%,OREB%,FTR,off_rating,def_rating,net_rating,season,pre_game_elo
0,28600001,1986-10-31 20:00:00,Thunder,1610612760,0,1.0,127,110,102.2976,0.622093,0.224834,0.448276,0.279070,124.147585,107.529404,16.618181,1987,1451.3865
1,28600001,1986-10-31 20:00:00,Trail Blazers,1610612757,1,0.0,110,127,100.1088,0.506494,0.199783,0.272727,0.480519,109.880450,126.861974,-16.981524,1987,1482.1389
2,28600002,1986-10-31 20:00:00,Pacers,1610612754,0,0.0,104,108,94.4640,0.469880,0.148205,0.318182,0.421687,110.094851,114.329268,-4.234417,1987,1433.3845
3,28600002,1986-10-31 20:00:00,76ers,1610612755,1,1.0,108,104,100.0320,0.523810,0.249920,0.461538,0.357143,107.965451,103.966731,3.998720,1987,1590.3137
4,28600003,1986-10-31 20:00:00,Wizards,1610612764,0,0.0,102,120,98.7264,0.435294,0.222838,0.400000,0.423529,103.315830,121.548036,-18.232205,1987,1478.1702


In [ ]:
# df.to_csv("data_with_elo.csv")

In [ ]:
# idx = df.groupby("season")["pre_game_elo"].idxmax()
# result_df = df.loc[idx, ["teamName", "pre_game_elo", "season"]]
# result_df.sort_values(by=["season"], ascending=[False])

,teamName,pre_game_elo,season
102672,Spurs,1791.04,2026
99590,Thunder,1804.93,2025
96872,Celtics,1778.15,2024
92530,Celtics,1717.24,2023
91608,Celtics,1759.07,2022
88832,Suns,1728.67,2021
85856,Bucks,1778.71,2020
83816,Rockets,1730.16,2019
80826,Rockets,1789.03,2018
78394,Warriors,1858.52,2017


In [19]:
df = pd.read_csv("data_with_elo.csv", index_col=0)

In [21]:
home_df = df[df["home"] == 1]
away_df = df[df["home"] == 0]

game_df = pd.merge(home_df, away_df, on='gameId', suffixes=('_home', '_away'))

In [23]:
game_df.columns

Index(['gameId', 'gameDateTimeEst_home', 'teamName_home', 'teamId_home',
       'home_home', 'win_home', 'points_home', 'opponentScore_home',
       'possessions_home', 'eFG_home', 'TO%_home', 'OREB%_home', 'FTR_home',
       'off_rating_home', 'def_rating_home', 'net_rating_home', 'season_home',
       'pre_game_elo_home', 'gameDateTimeEst_away', 'teamName_away',
       'teamId_away', 'home_away', 'win_away', 'points_away',
       'opponentScore_away', 'possessions_away', 'eFG_away', 'TO%_away',
       'OREB%_away', 'FTR_away', 'off_rating_away', 'def_rating_away',
       'net_rating_away', 'season_away', 'pre_game_elo_away'],
      dtype='str')